# 03 - Evaluate on the test set, compare with a baseline, measure efficiency, export

Run this **once** after training is final. The test set must not influence any decision you made earlier.

In [ ]:
import os, sys, json, glob
if not os.path.exists("InterviewAI"):
    !git clone -q https://github.com/Ishangupta-cyber/InterviewAI.git
%cd /content/InterviewAI
!git pull -q
sys.path.insert(0, "ml/src/resume_ner")
import data

OUT = "/content/drive/MyDrive/interview_ai_ml"   # splits, models and reports live here, so Colab resets cannot erase them
try:
    from google.colab import drive  # type: ignore  (this module only exists on Colab)
    drive.mount("/content/drive")
except Exception as e:   # Drive mount may be unavailable (e.g. running through VS Code)
    print("Drive not mounted (" + type(e).__name__ + "). Saving to /content instead; copy files out before the runtime ends.")
    OUT = "/content/interview_ai_ml"
os.makedirs(OUT, exist_ok=True)
print("outputs ->", OUT)

!pip -q install seqeval transformers datasets scikit-learn

In [ ]:
import time, ast, numpy as np, torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForTokenClassification
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from seqeval.scheme import IOB2

MODEL_DIR = f"{OUT}/resume_ner_model"
CHUNK, STRIDE = 150, 100
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR).eval()
labels = json.load(open(f"{MODEL_DIR}/labels.json"))
test = data.load_jsonl(f"{OUT}/test_resumes.jsonl")
print("test resumes:", len(test))

def predict_words(m, words):
    tags = [None] * len(words); start = 0
    while start < len(words):
        win = words[start:start + CHUNK]
        enc = tok(win, is_split_into_words=True, truncation=True, max_length=512, return_tensors="pt")
        with torch.no_grad():
            pred = m(**enc).logits.argmax(-1)[0].tolist()
        prev = None
        for pos, w in enumerate(enc.word_ids(0)):
            if w is not None and w != prev and tags[start + w] is None:
                tags[start + w] = labels[pred[pos]]
            prev = w
        if start + CHUNK >= len(words): break
        start += STRIDE
    return [t or "O" for t in tags]

y_true = [ex["tags"] for ex in test]
y_pred = [predict_words(model, ex["tokens"]) for ex in test]

## 1. Entity-level results (strict match: the whole span and label must be right)

In [ ]:
kw = dict(mode="strict", scheme=IOB2)
print(classification_report(y_true, y_pred, digits=3, **kw))
overall = {"precision": precision_score(y_true, y_pred, **kw), "recall": recall_score(y_true, y_pred, **kw),
           "f1": f1_score(y_true, y_pred, **kw)}
print("overall:", {k: round(v, 3) for k, v in overall.items()})

## 2. Baseline: the dictionary parser currently in the app

`backend/resume_service.py` finds skills by looking words up in a fixed list. Compare it with the model on **SKILLS**,
using token-level scores (the dataset labels whole skills paragraphs, so strict span matching would be unfair to the baseline).

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
tree = ast.parse(open("backend/resume_service.py").read())
vocab = next(ast.literal_eval(n.value) for n in tree.body
             if isinstance(n, ast.Assign) and getattr(n.targets[0], "id", "") == "SKILL_VOCAB")
vocab = {v.lower() for v in vocab}

def prf(true_bin, pred_bin):
    p, r, f, _ = precision_recall_fscore_support(true_bin, pred_bin, average="binary", zero_division=0)
    return round(p, 3), round(r, 3), round(f, 3)

gold = [t.endswith("SKILLS") for tags in y_true for t in tags]
model_pred = [t.endswith("SKILLS") for tags in y_pred for t in tags]
base_pred = [w.lower().strip(",.;:()") in vocab for ex in test for w in ex["tokens"]]
cmp = pd.DataFrame([{"system": "Dictionary baseline (current app)", **dict(zip(["precision", "recall", "f1"], prf(gold, base_pred)))},
                    {"system": "Fine-tuned DistilBERT", **dict(zip(["precision", "recall", "f1"], prf(gold, model_pred)))}])
display(cmp)

## 3. Error analysis (the part reports usually skip)

In [ ]:
from collections import Counter
conf = Counter()
for yt, yp in zip(y_true, y_pred):
    for a, b in zip(yt, yp):
        if a != b: conf[(a, b)] += 1
print("Most common mistakes (gold -> predicted):")
for (a, b), n in conf.most_common(12): print(f"  {n:5d}  {a:22s} -> {b}")

shown = 0
for ex, yp in zip(test, y_pred):
    bad = [(w, g, p) for w, g, p in zip(ex["tokens"], ex["tags"], yp) if g != p]
    if bad and shown < 3:
        print("\nExample mistakes:", bad[:8]); shown += 1

## 4. Efficiency: size, speed, and an int8-quantised version

In [ ]:
import os, copy
def size_mb(m):
    torch.save(m.state_dict(), "/tmp/_m.pt"); s = os.path.getsize("/tmp/_m.pt") / 1e6; os.remove("/tmp/_m.pt"); return round(s, 1)

def latency(m, runs=15):
    words = test[0]["tokens"][:400]
    predict_words(m, words)                       # warm-up
    t = time.perf_counter()
    for _ in range(runs): predict_words(m, words)
    return round((time.perf_counter() - t) / runs * 1000)

cpu = copy.deepcopy(model).cpu().eval()
q = torch.quantization.quantize_dynamic(copy.deepcopy(cpu), {torch.nn.Linear}, dtype=torch.qint8)
q_pred = [predict_words(q, ex["tokens"]) for ex in test]
eff = pd.DataFrame([
    {"model": "fp32", "params_M": round(sum(p.numel() for p in cpu.parameters()) / 1e6, 1), "size_MB": size_mb(cpu),
     "ms_per_400_words_CPU": latency(cpu), "test_F1": round(overall["f1"], 3)},
    {"model": "int8 (dynamic quant.)", "params_M": "-", "size_MB": size_mb(q),
     "ms_per_400_words_CPU": latency(q), "test_F1": round(f1_score(y_true, q_pred, **kw), 3)}])
display(eff)

## 5. Save the report files and the model

Download `resume_ner_model/` from Drive into `ml/models/resume_ner/` in your project (it is git-ignored: models are big).
Then the Django backend can load it with `ResumeNER` from `ml/src/resume_ner/predict.py`.

In [ ]:
report = {"overall_strict_entity_level": overall, "skills_token_level_comparison": cmp.to_dict("records"),
          "efficiency": eff.to_dict("records"), "test_resumes": len(test), "model": "distilbert-base-cased"}
json.dump(report, open(f"{OUT}/test_report.json", "w"), indent=1)
open(f"{OUT}/resume_ner_model/MODEL_CARD.md", "w").write(
    "# Resume NER (DistilBERT)\n\nTrained on the DataTurks Resume Entities for NER dataset (see notebook 01).\n"
    "Labels: " + ", ".join(labels) + "\n\nStrict entity-level test F1: " + str(round(overall["f1"], 3)) +
    "\n\nLimits: small training set (hundreds of resumes, indeed.com style); may not generalise to other layouts, "
    "languages or non-IT resumes. Do not use it to rank or reject candidates.\n")
print(json.dumps(report, indent=1)[:1500])